In [1]:
%pip install sahi ultralytics opencv-python

In [2]:
import os 

video_path = 'people_in_üsküdar.mp4'
if os.path.exists(video_path):
    print("Video başarıyla bulundu!")
else:
    print("Video bulunamadı, lütfen dosya yolunu kontrol et.")

In [ ]:
import cv2 as cv 
from sahi.predict import get_sliced_prediction
from sahi import AutoDetectionModel

detection_model = AutoDetectionModel.from_pretrained(
    model_type='ultralytics',
    model_path='yolov8m.pt',
    confidence_threshold=0.3,
    device="cuda:0"
)     

cap = cv.VideoCapture(video_path)
width  = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv.CAP_PROP_FPS))
out = cv.VideoWriter('output_üsküdar.mp4', cv.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

print("işlem başlıyor...")

frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break 
    

    result = get_sliced_prediction(
        frame,
        detection_model,
        slice_height=512,
        slice_width=512,
        overlap_height_ratio=0.2,
        overlap_width_ratio=0.2
    )

    # kutuları manuel çiz
    frame_with_results = frame.copy()
    for prediction in result.object_prediction_list:
        bbox = prediction.bbox
        x1, y1, x2, y2 = int(bbox.minx), int(bbox.miny), int(bbox.maxx), int(bbox.maxy)
        label = f"{prediction.category.name} {prediction.score.value:.2f}"

        cv.rectangle(frame_with_results, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv.putText(frame_with_results, label, (x1, y1 - 10),
                   cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    out.write(frame_with_results)

cap.release()
out.release()
print("işlem tamamlandı! output_üsküdar.mp4 dosyası oluşturuldu")